In [3]:
import os
import pandas as pd
import numpy  as np
from datetime import datetime

In [4]:
############################################ 
# mrt --> imdb
############################################
# data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
# import_file = os.path.join(data_path, "mrt_xref_titles_principals__enriched.csv")
data_path = r"G:/My Drive/casablanca/rpi/prod/interfaces/rpi20003_twb__cast_and_crew/export/current/"
import_file = os.path.join(data_path, "mrt_xref_titles_principals__enriched.csv")

imdb = pd.DataFrame()
imdb = pd.read_csv(import_file, encoding="utf-8", index_col=0)
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'imdb: ', imdb.shape)

C:\Users\cwern\AppData\Local\Temp\ipykernel_32104\1968437852.py:10: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  imdb = pd.read_csv(import_file, encoding="utf-8", index_col=0)


2026-07-01 12:07:36 imdb:  (4665646, 31)


In [5]:
#################################################
#remove rows not needed  = startgin point for all aggregations to come 
#################################################

X = imdb[imdb['category']!= "production_designer"] 
X = X[X['category']!= "archive_sound"] 
X = X[X['category']!= "casting_director"]
#X = X[X['category']!= "archive_footage"]
print(X.shape, imdb.shape)

#################################################
#remove columns not needed 
#################################################

X = X.drop('primaryProfession', axis=1)
print(X.shape, imdb.shape)
#X.sort_values(by=['tconst', 'averageRating'], ascending=False)

#pd.set_option('max_columns', 30)
X[X['tconst'] == "tt13833688"].head(3)

(4665646, 31) (4665646, 31)
(4665646, 30) (4665646, 31)


,tconst,primaryName_Dir0,tconst_your_rating,Release Date,tconst_imdb_rating,tconst_imdb_num_votes,nconst,primaryName,birthYear,deathYear,...,startYear,endYear,runtimeMinutes,genres,flg_doc,primaryTitle_year,rate_diff,age_crew,last_refresh,RN
2634213,tt13833688,Darren Aronofsky,5.0,2022-12-21,7.6,302511,nm0000409,Brendan Fraser,1968,0,...,2022,0,117,Drama,Fiction,The Whale (2022),-2.6,54,2026-06-30 17:05:43,2
2634214,tt13833688,Darren Aronofsky,5.0,2022-12-21,7.6,302511,nm0004716,Darren Aronofsky,1969,0,...,2022,0,117,Drama,Fiction,The Whale (2022),-2.6,53,2026-06-30 17:05:43,4
2634215,tt13833688,Darren Aronofsky,5.0,2022-12-21,7.6,302511,nm0206154,Jeremy Dawson,0,0,...,2022,0,117,Drama,Fiction,The Whale (2022),-2.6,,2026-06-30 17:05:43,1


In [6]:
############################################ 
### import dwh_titles__editor_lists
############################################
data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
import_file = os.path.join(data_path, "dwh_titles__editor_lists.csv")

dwh_titles__editor_lists = pd.DataFrame()
dwh_titles__editor_lists = pd.read_csv (import_file, encoding='utf-8', index_col=0)
print(dwh_titles__editor_lists.shape)
dwh_titles__editor_lists.tail(3) 

(5610, 11)


,tconst,Title,Description,Directors,Genres,Release Date,URL,festival_name,festival_year,festival_url,rec_cnt
5607,tt27817389,The Last Viking,NaN,Anders Thomas Jensen,"Comedy, Crime, Drama",2025-10-09,https://www.imdb.com/title/tt27817389/,YTD,2025,https://www.kulturinfo.ch/kino/index.html,1
5608,tt27714581,Sentimental Value,NaN,Joachim Trier,Drama,2025-08-20,https://www.imdb.com/title/tt27714581/,YTD,2025,https://www.kulturinfo.ch/kino/index.html,1
5609,tt35231039,Kokuho,NaN,Sang-il Lee,Drama,2025-11-21,https://www.imdb.com/title/tt35231039/,YTD,2025,https://www.kulturinfo.ch/kino/index.html,1


In [7]:
############################################ 
### merge title with festival
############################################        
Xfst = pd.merge(    X
                  , dwh_titles__editor_lists
                  , how = 'inner', left_on="tconst", right_on="tconst"
       #         , how = 'right', left_on="tconst", right_on="tconst"
                 )
# Xfst.to_csv('data/Xfst.csv' , header=1 , encoding='utf-8')

In [8]:
############################################ 
### export result  -- 
############################################ 
"""
import os
data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
output_file = os.path.join(data_path, "xto_flat_titles_principals__festival_enriched.csv")

Xfst.to_csv(output_file, encoding="utf-8")
"""

'\nimport os\ndata_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"\noutput_file = os.path.join(data_path, "xto_flat_titles_principals__festival_enriched.csv")\n\nXfst.to_csv(output_file, encoding="utf-8")\n'

In [9]:
############################################ 
# TCONST  - Principal  Enrichment/Aggregations
# duplicate, so titels have both ratings (imdb, profile_1)
############################################ 

import numpy as np
# Xfst2 = pd.DataFrame({'Xfst':['tconst','nconst'],})
# Xfst2 = pd.DataFrame({'Xfst':['tconst','nconst'],})
Xfst2 = Xfst.copy()   # panda refernces tables only, so explicit copy required
  
Xfst2['rating_category']           = 'IMDb'
Xfst2['tconst_num_votes']          = np.where(Xfst2['tconst_imdb_num_votes'].isnull(), 0, Xfst2['tconst_imdb_num_votes'])   
Xfst2['tconst_rating']             = np.where(Xfst2['tconst_imdb_rating'].isnull(), np.nan, Xfst2['tconst_imdb_rating'] )   
Xfst2.head(3)

Xfst3 = Xfst.copy()
  
Xfst3['rating_category']           = 'Profile_1'
#Xfst3['tconst_num_votes']          = 1   
#Xfst3['tconst_rating']             = np.where(Xfst3['tconst_your_rating'].isnull(), 0, Xfst3['tconst_your_rating'] )
Xfst3['tconst_num_votes']       = np.where(Xfst3['tconst_your_rating'].isnull(), 0, 1)
Xfst3['tconst_rating']          = np.where(Xfst3['tconst_your_rating'].isnull(), np.nan, Xfst3['tconst_your_rating'] )
Xfst3.head(3)
 
mrt_titles__per_rating_category = pd.concat([Xfst2, Xfst3], ignore_index=True) 
mrt_titles__per_rating_category.tail(4)
 

,tconst,primaryName_Dir0,tconst_your_rating,Release Date_x,tconst_imdb_rating,tconst_imdb_num_votes,nconst,primaryName,birthYear,deathYear,...,Genres,Release Date_y,URL,festival_name,festival_year,festival_url,rec_cnt,rating_category,tconst_num_votes,tconst_rating
131318,tt9897230,Luka Beradze,NaN,NaN,7.3,61,nm3958502,Nino Chichua,0,0,...,Documentary,2023-07-04,https://www.imdb.com/title/tt9897230/,KVIFF,2023,https://www.kviff.com,1,Profile_1,0,NaN
131319,tt9897230,Luka Beradze,NaN,NaN,7.3,61,nm6150255,Anna Khazaradze,0,0,...,Documentary,2023-07-04,https://www.imdb.com/title/tt9897230/,KVIFF,2023,https://www.kviff.com,1,Profile_1,0,NaN
131320,tt9897230,Luka Beradze,NaN,NaN,7.3,61,nm8067036,Luka Beradze,0,0,...,Documentary,2023-07-04,https://www.imdb.com/title/tt9897230/,KVIFF,2023,https://www.kviff.com,1,Profile_1,0,NaN
131321,tt9897230,Luka Beradze,NaN,NaN,7.3,61,nm9783168,Ioseb 'Soso' Bliadze,0,0,...,Documentary,2023-07-04,https://www.imdb.com/title/tt9897230/,KVIFF,2023,https://www.kviff.com,1,Profile_1,0,NaN


In [10]:
# lst = ['nm0000001', 'nm0000002']
# N_unique =N_unique.query('nconst in @lst')
# N_unique.to_csv('data/nconst.csv' , header=0 , encoding='utf-8')

# T --> cln_imdb__xref_titles_principals_filtered

import os
data_path = r"G:/My Drive/casablanca/rpi/prod/interfaces/rpi20003_twb__cast_and_crew/export/current/"
output_file = os.path.join(data_path, "mrt_titles__per_rating_category.csv")
 

mrt_titles__per_rating_category.to_csv(output_file, encoding="utf-8")



In [11]:
#################################################
# NCONST  - Principal Aggregations
# Distinct NCONST aggregation
#################################################
import numpy as np

N_avg = X.pivot_table(  index   = ( 'nconst', 'primaryName_year', 'cat_cln')
                   , aggfunc = [ np.mean]                   
                   , values = ["tconst_imdb_rating", "tconst_your_rating"]
      ##             , fill_value = 0
             )
N_sum = X.pivot_table(  index   = ( 'nconst', 'primaryName_year', 'cat_cln')
                   , aggfunc = [ np.sum]                   
                   , values = [ "tconst_imdb_num_votes"]
                   , fill_value = 0
             )
N_max = X.pivot_table(  index   = ( 'nconst', 'primaryName_year', 'cat_cln')
                   , aggfunc = [ np.max]                   
                   , values = [ "startYear"]
                   , fill_value = 0
             )
N_cnt = X.pivot_table(  index   = ( 'nconst', 'primaryName_year', 'cat_cln')
                   , aggfunc = [ 'count']  
                   , values = ["tconst"]
                   , fill_value = 0
             )

N_unique = pd.merge(   N_sum
                     , N_avg 
                     , how = 'inner', left_on=  [ 'nconst', 'primaryName_year', 'cat_cln'], right_on= [ 'nconst', 'primaryName_year', 'cat_cln']
                 )
N_unique = pd.merge(   N_unique
                     , N_cnt
                     , how = 'inner', left_on= [ 'nconst', 'primaryName_year', 'cat_cln'], right_on= [ 'nconst', 'primaryName_year', 'cat_cln']
                 )
N_unique = pd.merge(   N_unique
                     , N_max
                     , how = 'inner', left_on= [ 'nconst', 'primaryName_year', 'cat_cln'], right_on= [ 'nconst', 'primaryName_year', 'cat_cln']
                 )

#format column names - grouping created by pivo table
N_unique.columns = ['_'.join(str(s).strip() for s in col if s) for col in N_unique.columns]

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


#reset index
N_unique.reset_index(inplace=True)


N_unique.rename(columns = {  "sum_tconst_imdb_num_votes": "tconst_imdb_num_votes"
                           , "mean_tconst_imdb_rating":   "tconst_imdb_rating"
                           , "mean_tconst_your_rating":   "tconst_your_rating"
                           , "count_tconst":              "cnt_tconst"
                           , "amax_startYear":            "max_startYear"}
                           , inplace = True)
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'N_unique: ', N_unique.shape)
N_unique.head(3)

C:\Users\cwern\AppData\Local\Temp\ipykernel_32104\129577394.py:7: FutureWarning: The provided callable <function mean at 0x000001C2B94D4F40> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  N_avg = X.pivot_table(  index   = ( 'nconst', 'primaryName_year', 'cat_cln')
C:\Users\cwern\AppData\Local\Temp\ipykernel_32104\129577394.py:12: FutureWarning: The provided callable <function sum at 0x000001C2B94A7B00> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  N_sum = X.pivot_table(  index   = ( 'nconst', 'primaryName_year', 'cat_cln')
C:\Users\cwern\AppData\Local\Temp\ipykernel_32104\129577394.py:17: FutureWarning: The provided callable <function max at 0x000001C2B94D4540> is currently using DataFrameGroupBy.max. In a future version of pandas,

2026-07-01 12:09:12
2026-07-01 12:09:13 N_unique:  (1698857, 8)


,nconst,primaryName_year,cat_cln,tconst_imdb_num_votes,tconst_imdb_rating,tconst_your_rating,cnt_tconst,max_startYear
0,nm0000001,Fred Astaire (1899 - 1987),actor,298632,6.692308,NaN,39,1981
1,nm0000001,Fred Astaire (1899 - 1987),self,19643,6.905556,NaN,18,2021
2,nm0000002,Lauren Bacall (1924 - 2014),actor,982100,6.480952,6.0,42,2012


In [12]:
# lst = ['nm0000001', 'nm0000002']
# N_unique =N_unique.query('nconst in @lst')
# N_unique.to_csv('data/nconst.csv' , header=0 , encoding='utf-8')
# T --> cln_imdb__xref_titles_principals_filtered

import os

data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
output_file = os.path.join(data_path, "mrt_principals__aggregated.csv")
N_unique.to_csv(output_file, encoding="utf-8")

In [13]:
N_unique.head(3)

,nconst,primaryName_year,cat_cln,tconst_imdb_num_votes,tconst_imdb_rating,tconst_your_rating,cnt_tconst,max_startYear
0,nm0000001,Fred Astaire (1899 - 1987),actor,298632,6.692308,NaN,39,1981
1,nm0000001,Fred Astaire (1899 - 1987),self,19643,6.905556,NaN,18,2021
2,nm0000002,Lauren Bacall (1924 - 2014),actor,982100,6.480952,6.0,42,2012


In [14]:
# duplicate, so principals  have both ratings (imdb, profile_1)



N2 = N_unique.copy()   # panda refernces tables only, so explicit copy required
  
N2['rating_category']           = 'IMDb'
N2['tconst_num_votes']          = np.where(N2['tconst_imdb_num_votes'].isnull(), 0, N2['tconst_imdb_num_votes'] )  
N2['tconst_rating']             = np.where(N2['tconst_imdb_rating'].isnull(), np.nan, N2['tconst_imdb_rating'] )   
N2.head(3)

N3 = N_unique.copy()
  
N3['rating_category']           = 'Profile_1'
N3['tconst_num_votes']          = np.where(N3['tconst_your_rating'].isnull(), 0, 1 )
N3['tconst_rating']             = np.where(N3['tconst_your_rating'].isnull(), np.nan, N3['tconst_your_rating'] )
N3.head(3)
 


mrt_principals__per_rating_category = pd.concat([N2, N3], ignore_index=True) 
# nconst_duplicate.to_csv('data/nconst_duplicate.csv' , header=1 , encoding='utf-8')


In [15]:
# lst = ['nm0000001', 'nm0000002']
# N_unique =N_unique.query('nconst in @lst')
# N_unique.to_csv('data/nconst.csv' , header=0 , encoding='utf-8')

# T --> cln_imdb__xref_titles_principals_filtered

import os
 
# data_path = r"G:/My Drive/casablanca/rpi/prod/interfaces/rpi2000_twb__principals/export/current/"
data_path = r"G:/My Drive/casablanca/rpi/prod/interfaces/rpi20003_twb__cast_and_crew/export/current/"
output_file = os.path.join(data_path, "mrt_principals__per_rating_category.csv")
# output_file = os.path.join(data_path, "rpi20002_twb__principals_per_rating_category.csv")
mrt_principals__per_rating_category.to_csv(output_file, encoding="utf-8")